In [1]:
import os
from typing import List,TypedDict
from IPython.display import display,Markdown
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from google import genai
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings

/tmp/ipykernel_55434/342637125.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,TextLoader


In [2]:
load_dotenv()
if not os.getenv('Google_API_Key'):
    raise ValueError('Google API Key is not Found')

In [3]:
client=genai.Client(api_key=os.getenv('Google_API_Key'))

for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/lyri

In [4]:
for e in client.models.list():
    if 'embedContent' in e.supported_actions:
        print(e.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [5]:
model='models/gemini-flash-lite-latest'
embed_model='models/gemini-embedding-2'
CHROMA_DIR='/home/sinanshareef/Documents/Artificial Intelligence/Generative AI/LangGraph/C_RAG/Hospital_RAG/vector_database'
DOC_DIR='/home/sinanshareef/Documents/Artificial Intelligence/Generative AI/LangGraph/C_RAG/Hospital_DOC'
Retrieve_k=4
Max_Retries=2


In [6]:
def loading_docs(folder):
    docs=[]
    if not os.path.exists(folder):
        raise FileNotFoundError(folder,'File not found')
    for file in os.listdir(folder):
        path=os.path.join(folder,file)
        try:
            if file.endswith('.pdf'):
                loader=PyPDFLoader(path)
            elif file.endswith('txt'):
                loader=TextLoader(path,encoding='utf-8')
            else:
                continue

            loaded=loader.load()
            for d in loaded:
                d.page_content=d.page_content.replace('■','₹')
            docs.extend(loaded)
        except Exception as e:
            print(f'File skipped as {e}')
    return docs 
docs=loading_docs(DOC_DIR)          


In [7]:
print(len(docs))

4


In [8]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=380,
    chunk_overlap=140
)

chunks=splitter.split_documents(docs)
len(chunks)

59

In [9]:
def get_vectordb():
    embeddings=GoogleGenerativeAIEmbeddings(model=embed_model,api_key=os.getenv('Google_API_Key'))
    if os.path.exists(CHROMA_DIR):
        store=Chroma(
            persist_directory=CHROMA_DIR,
            embedding_function=embeddings,
        )
        if store._collection.count()>0:
            return store
    return Chroma.from_documents(
        chunks,
        embedding=embeddings,
        persist_directory=CHROMA_DIR,
    )

vector_store=get_vectordb()

In [10]:
retrieve=vector_store.as_retriever(search_kwargs={'k':Retrieve_k})
llm=ChatGoogleGenerativeAI(model=model,temperature=0,api_key=os.getenv('Google_API_Key'),max_retries=1,timeout=60)

In [11]:
class GraphState(TypedDict):
    question:str
    documents:list[dict]   #Content and Source both are in str
    generation:str
    retry_count:int

In [12]:
def retrieval(state: GraphState) ->GraphState:
    print('retrieving...')
    docs=retrieve.invoke(state['question'])
    documents=[{'content':doc.page_content,
               'source':doc.metadata.get('source','unknown')}
               for doc in docs
               ]
    return {**state,'documents':documents}

In [13]:
def grade_documents(state:GraphState)->GraphState:
    print('Grading...')
    grade_prompt = ChatPromptTemplate.from_template(
            "Question: {question}\nDocument: {documents}\n\n"
            "Does this document contain information relevant to the question? "
            "Answer only 'yes' or 'no'."
        )
    grader=grade_prompt|llm|StrOutputParser()

    retrievee=[]
    for doc in state['documents']:
        try:
            result=grader.invoke({'question':state['question'],'documents':doc['content']})
        except Exception as e:
            print(f'Grader call failed, keeping document: {e}')
            retrievee.append(doc)
            continue

        if 'yes' in result.lower():
            retrievee.append(doc)
    return{**state,'documents':retrievee}

In [14]:
def transform_query(state:GraphState)->GraphState:
    print('Rewrite Query')
    rewrite_prompt = ChatPromptTemplate.from_template(
            "The original question did not retrieve good results:\n{question}\n\n"
            "Rewrite it to be clearer and more specific for document search. "
            "Return only the rewritten question."
        )
    rewrite=rewrite_prompt|llm|StrOutputParser()

    new_question=rewrite.invoke({'question':state['question']})
    return {**state,'question':new_question,'retry_count':state['retry_count']+1}

In [15]:
def generate(state:GraphState)->GraphState:
    print('Generating')
    gen_prompt = ChatPromptTemplate.from_template(
            "Answer the question using only the context below.\n"
            "If the answer is not in the context, say you don't know.\n"
            "Answer directly and concisely with only the needed information; "
            "do not mention the context, documents or sources.\n\n"
            "Context:\n{context}\n\nQuestion:\n{question}"
        )
    context='\n\n'.join([doc['content'] for doc in state['documents']])
    chain=gen_prompt|llm|StrOutputParser()
    answer=chain.invoke({'context':context,'question':state['question']})
    return {**state,'generation':answer}

In [16]:
def next_step(state:GraphState)->GraphState:
    print('Deciding the Next Step')
    if len(state['documents'])==0 and state['retry_count'] < Max_Retries:
        return "transform_query"

    return 'generate'

In [17]:
workflow=StateGraph(GraphState)
workflow.add_node('retrieval',retrieval)
workflow.add_node('grade_documents',grade_documents)
workflow.add_node('transform_query',transform_query)
workflow.add_node('generate',generate)

workflow.set_entry_point("retrieval")
workflow.add_edge("retrieval","grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    next_step,
    {"transform_query":"transform_query","generate":"generate"}
)
workflow.add_edge("transform_query","retrieval")
workflow.add_edge("generate",END)

app=workflow.compile()

In [18]:
if __name__ == "__main__":
    while True:
        question=input("\n Ask Something (or quite)")
        if question.lower()=='quit':
            break
        if not question.strip():
            continue
        try:
            output=app.invoke(
                {
                    'question':question,
                    'documents':[],
                    'generation':"",
                    "retry_count":0
                }
            )

            print("\n Answer: ",output['generation'])

        except Exception as e:
            print(f'Something went wrong: {e}')

retrieving...
Grading...
Deciding the Next Step
Generating

 Answer:  24 Riverside Road, Greenfield Junction, Kochi, Kerala 6820XX
retrieving...
Grading...
Deciding the Next Step
Generating

 Answer:  I don't know.
